## Sélection de variables plus importantes

In [11]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_validate
from  sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import r2_score, f1_score
from sklearn.preprocessing import (
    OneHotEncoder, TargetEncoder, OrdinalEncoder,
    StandardScaler, RobustScaler,
    FunctionTransformer
)

### Création de variables très importantes

In [12]:
new_path = os.path.abspath(os.getcwd())
path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), "models", "best_model_tuned.pkl")

random_model = joblib.load(path)
importance = random_model["random_model"].feature_importances_

train_data = pd.read_csv("../data/processed_dataset.csv")
test_data = pd.read_parquet("../data/test_data.parquet")

columns = train_data.drop(["ID_Employe", "Satisfait"], axis=1).select_dtypes(include=["number"]).columns

classement = pd.DataFrame({
    "Variables": columns,
    "Feature_Importance": importance
}).sort_values(ascending=False, by="Feature_Importance")

feature_importance_cols = classement.nlargest(7, "Feature_Importance")["Variables"]

X_train_all_features = train_data[columns]
X_train_importance = train_data[feature_importance_cols]
y_train = train_data["Satisfait"]

print(feature_importance_cols)
print()
print(X_train_importance.columns)

8      Equilibre_Vie_Travail
9       Satisfaction_Salaire
4           Heures_Formation
10           Nombre_Absences
3     Heures_Supplementaires
2        Salaire_Mensuel_BIF
0                        Age
Name: Variables, dtype: str

Index(['Equilibre_Vie_Travail', 'Satisfaction_Salaire', 'Heures_Formation',
       'Nombre_Absences', 'Heures_Supplementaires', 'Salaire_Mensuel_BIF',
       'Age'],
      dtype='str')


### Mise en niveau de données de test

In [13]:
# Séparation de X_test et y_test
X_test_all_features = test_data[columns]

X_test_importance = test_data[feature_importance_cols]
y_test = test_data["Satisfait"]

X_test_importance

,Equilibre_Vie_Travail,Satisfaction_Salaire,Heures_Formation,Nombre_Absences,Heures_Supplementaires,Salaire_Mensuel_BIF,Age
0,1.0,1.0,0.655394,-0.487980,-1.289932,-0.187169,0.203711
1,3.0,4.0,0.860110,0.696083,-0.672980,-0.920043,-1.590189
2,3.0,1.0,1.211052,0.696083,-0.056028,-0.840417,-0.479679
3,2.0,1.0,-0.221961,1.643334,-1.289932,0.652730,0.024057
4,3.0,1.0,-1.070071,0.222458,0.120245,-0.784460,-1.419341
...,...,...,...,...,...,...,...
123,3.0,4.0,-0.104980,-1.435231,0.913469,1.419495,0.801677
124,1.0,2.0,-0.280451,0.104051,-1.642476,-0.917386,0.545406
125,3.0,3.0,0.421432,-0.014355,-1.113660,1.988563,1.485068
126,3.0,2.0,-1.596484,1.761740,0.913469,0.012349,-0.906798


### Comparaison de modèles via la sélection de variables très important et toutes les variables

In [14]:
# Entrainement sur toutes les variables
all_features_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=7,
    n_jobs=-1
)

all_features_model.fit(X_train_all_features, y_train)
y_predict_all_features = all_features_model.predict(X_test_all_features)

In [15]:
# Entrainement les variables plus importantes
importance_features_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=7,
    n_jobs=-1
)

importance_features_model.fit(X_train_importance, y_train)
y_predict_importance_features = importance_features_model.predict(X_test_importance)

In [16]:
# Score de deux modèles
print(f"Score pour all_features_model : {f1_score(y_test, y_predict_all_features)}")
print(f"Score pour importance_features_model : {f1_score(y_test, y_predict_importance_features)}")

Score pour all_features_model : 0.8383233532934131
Score pour importance_features_model : 0.8214285714285714


In [17]:
file_path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), "data", "artefacts.joblib")
artefacts = joblib.load(file_path)

mean = {
    col: X_train_importance[col].mean() for col in X_train_importance.select_dtypes(include=['number']).columns}
modes = {
    col: X_train_importance[col].mode()[0] for col in X_train_importance.select_dtypes(include=['object']).columns}

artefacts = {
    'random_model': importance_features_model,
    'one_hot_encoder': artefacts["one_hot_encoder"],
    'ordinal_encoder': artefacts["ordinal_encoder"],
    'standard_scaler': artefacts["standard_scaler"],
    'mean': mean,
    'modes': modes,
    'columns': list(X_train_importance.columns)
}

path = os.path.join(os.path.dirname(os.path.abspath(os.getcwd())), "models", "final_model.pkl")
joblib.dump(artefacts, path)

print(f"Modèle et prétraitements sauvegardés dans {path}")

Modèle et prétraitements sauvegardés dans /home/luckson/AI-Project/employee_satisfaction/models/final_model.pkl
